<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c36-header.png" alt="Nextia Learning · Text Classification and Routing: Send Each Message to the Right Place" width="100%">

# Solution: Ask for a decision, not for text

**[Ask for a decision, not for text](https://learning.nextia-ai.com/courses/text-routing/m06/ask-for-a-decision-not-for-text/)** · Text Classification and Routing: Send Each Message to the Right Place · Module 6, lesson 1 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** get a route **and a probability** from a small local language model, by scoring each route's name instead of letting the model write text, and to check if those probabilities are true.

| | |
|---|---|
| **Time** | About 40 minutes. The model download is about 1.5 GB, once. |
| **Needs** | An internet connection for the setup cells and the model download from Hugging Face. About 4 GB of free memory. No account and no key. |
| **You should know** | Small local LLMs, from [Small local LLMs](https://learning.nextia-ai.com/courses/text-routing/m05/small-local-llms/). Calibration, from the machine learning course (helpful). |
| **Tested** | Python 3.12 with pandas 3.1.1, scikit-learn 1.9.1 and matplotlib 3.11.2, transformers 5.22.2 and PyTorch 2.15.0 (CPU), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/text-routing/m06/ask-for-a-decision-not-for-text/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C36/M06-L01-ask-for-a-decision-not-for-text/ask-for-a-decision-not-for-text-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: pandas 3.1.1, scikit-learn 1.9.1, transformers 5.22.2, torch 2.15.0, and matplotlib for the charts. It needs an internet connection and takes about a minute the first time. PyTorch is large (about 200 MB to download): the first run of this cell can take 2 minutes.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 scikit-learn==1.9.1 matplotlib==3.11.2 transformers==5.22.2 torch==2.15.0
import warnings; warnings.filterwarnings("ignore")
import pandas; print("pandas", pandas.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)
import transformers; print("transformers", transformers.__version__)
import torch; print("torch", torch.__version__)

## Setup: get the practice data

Run the next cell. It downloads the Larkfield inbox of this course (about 9 MB) into a folder `larkfield-inbox/data`, and the course's helper `routing.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: 10000 messages (6000 train, 2000 validation, 2000 test), all synthetic.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-inbox` (in Colab: the **Files** panel on the left), then run the cell again.

The Larkfield inbox is synthetic: we made it for this course. It has no real people. The answers of the hosted models were recorded once, so you need no key.

In [ ]:
# Setup: download the course data into larkfield-inbox/ and check it.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C36/"
FILES = {
    "data/corrections.csv": "be3088aecd9b7ba5d07ea222b79e06a476806eba81bc80c2f33da695878e5cc0",
    "data/costs.json": "9a2d566f062c3382b8884ef07756f886179c69a3525d80e3d4c15c0c4fd4507e",
    "data/dataset-card.md": "fce0d2e602002a2ea2c6fc26c1d2ab9c3d1a30906f40fa4a426180c3639e99af",
    "data/double_labelled.csv": "ba4a8dadc3c1ba8d48af893124bd33fb7a895f917a2add211486e3dbd353dfc9",
    "data/guidelines.md": "8ab83de04998b9017b1aabce00be04084e989af75aff5a9abdd760f4d3cb24b2",
    "data/launch_labels_sample.csv": "600c689325b7c14e56f4a27e92e395eb0b2e965411b1abf9d724720c8b485029",
    "data/launch_month.csv": "7ef3cc2c5fa910023c6330c1e895744358846ce451e551b86be8c9abe1c4289f",
    "data/messages.csv": "266ca23c0551249fc114fc74253103ea8602a04be1fe31d1b7f91bc94ce6dd5e",
    "data/old_queue.csv": "e3cde5d19d3ebe2e6e0e067693c76796c0abbf89df0dc93dc310ad3fe42a5cae",
    "data/predictions/embeddings-newer.csv": "3c7ffe5b5371ca60c3b6bedc17b307142bcfd77f06c97db7ed267b703a50a8d2",
    "data/predictions/embeddings-newer.json": "dcd537f2971b9a48b5354cc9fbb48b87689b83b3a479da9acd119701c2fc19f6",
    "data/predictions/embeddings-older.csv": "a06c490ac17cbc1c3c75380f6cdf7775483de92961cdbb78b2a7914256926539",
    "data/predictions/embeddings-older.json": "db658bbd13861529122807726b6b41e1da9c70e3b55aba55cce9e868a795e8b4",
    "data/predictions/encoder-finetuned.csv": "abced2276671bfd413319573907a48c60952310e50978dba96c0dde0332b7d39",
    "data/predictions/encoder-finetuned.json": "1f7c89271a199c1d659bdc29d8ac0592558a1424218bd0519bbbde9840212b94",
    "data/predictions/gliclass-zero-shot.csv": "2c22e5036de94a376f7c0ccf22603c4d7c624135c8a4965123c9492748814008",
    "data/predictions/gliclass-zero-shot.json": "95ad2acafd48ad2c24516333dba7e78a3eabc7ca616c4053ce2f144f56cbaaec",
    "data/predictions/llm-few-shot.csv": "bcdaee3550f78ebb672465314bbcc9ad26283aaea69265a979739cf6a5f965e2",
    "data/predictions/llm-few-shot.json": "104b446894a0450a7e02fcd07c6db44f3e1a5486002b388374ce6dae4a744a12",
    "data/predictions/llm-zero-shot.csv": "5518e2b668a110b572687313bc13bd689774dff96c0a1614e1cd92e14ddf9ce7",
    "data/predictions/llm-zero-shot.json": "9a72ce600d0d858b00dd3b37247a305f9e8b3fc2fef3b8dbe60c0871b0c1b355",
    "data/predictions/local-llm-prompted.csv": "5bb0d84f8ee9b5086818d3525e1da3f8172b38b7875b10d1d3021b3ad5889f33",
    "data/predictions/local-llm-prompted.json": "ae328379d5ea294b3b59845ee143f24776787e748bc14c165b5e8e9ed4ff7158",
    "data/predictions/local-llm-scores.csv": "f85f570075ea9288fd14a6c3955d4310ba740814f3de472e8a263cd4f9682a5f",
    "data/predictions/local-llm-scores.json": "67a7e07ce065934a947f59c1c36d416eec76f8827f01500ce82ba6e6c4d9b99e",
    "data/predictions/nli-zero-shot.csv": "ecafb001370e5cf2904ce685683beafc86a2c290cd1da80438abb9912d099d3c",
    "data/predictions/nli-zero-shot.json": "e271f3b12ad86c6308a74c0b5f8b6bf80c6534cf18c41af5c86c29b184ab53cd",
    "data/prices.json": "9e98f9056d9d1e53d76a039da9b10d2b4fa54c8a5b35eb6cae32a2abdee7af38",
    "data/recordings/README.md": "d99eb50666045cc2ccf596e100103244fe1513b488fce46e26462a2666053ffc",
    "data/recordings/llm-few-shot.jsonl": "259a93f1d6dcbcb167cd45796c8e09a914e56c7741f14bd0aee53c996e8349d4",
    "data/recordings/llm-labels.jsonl": "cd6c670a23ee78cd3d8b51e5907ebae9d86448aa6236767c11d55227616fa8cc",
    "data/recordings/llm-reasoning.jsonl": "fc8ae631121f3240ce378df429af767ac52ea6e1560aa409de09ffc99b092300",
    "data/recordings/llm-zero-shot.jsonl": "957edfb8811e177c83bf23772f0da561659cf50728ffc95597f85cd7680aed86",
    "data/rules.json": "6f95d5e7b87869ec7975b198a3777cdd96a0db6a869554d922889479c87b2066",
    "routing.py": "8de6562d53415e5c3d8e74f44df9e8838cb71085011b29244187bde5a6f4d93c",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-inbox").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and sha256(path) == expected:
        continue
    local = HERE / name  # a copy next to the notebook, if there is one
    if local.exists():
        shutil.copy(local, path)
    else:
        download(LABS + name, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-inbox and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
import csv
with open("data/messages.csv", encoding="utf-8") as f:
    splits = [row["split"] for row in csv.DictReader(f)]
print(f"Ready: {len(splits)} messages ({splits.count('train')} train, {splits.count('validation')} validation, "
      f"{splits.count('test')} test), all synthetic.")

## Setup: helpers

The next cell loads pandas as `pd`, numpy as `np` and the course helper as `rt`, reads the three parts of the inbox (`train`, `validation`, `test`), and defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the data, the course helper, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import routing as rt
warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 160)

messages = pd.read_csv("data/messages.csv")
train = messages[messages["split"] == "train"].reset_index(drop=True)
validation = messages[messages["split"] == "validation"].reset_index(drop=True)
test = messages[messages["split"] == "test"].reset_index(drop=True)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Load a small model

`Qwen/Qwen3-0.6B` is a small instruction model (0.6 billion parameters, Apache-2.0) that runs on a laptop CPU. In the last module, we asked such a model to **write** its answer. Now we ask it a different question: *how likely is each route's name as the answer?*

The next cell loads the model in 32-bit floats (about 2.4 GB of memory) and builds the prompt for one message.

In [ ]:
import time, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
torch.set_num_threads(4)
tok = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B", revision="c1899de289a04d12100db370d81485cdf75e47ca")
lm = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-0.6B", revision="c1899de289a04d12100db370d81485cdf75e47ca", dtype=torch.float32).eval()
SYSTEM = "You route customer messages for Larkfield, an online shop for home and garden products. Messages can be in English or French. Answer with the name of exactly one route.\nRoutes:\n- orders: changes to an order before it ships: change or cancel an order, a missing order confirmation\n- delivery: where a parcel is, late or missed deliveries, a new delivery day, a parcel marked delivered but not received\n- returns_refunds: returns, exchanges, items that arrived damaged or wrong, the status of a refund after a return\n- billing: charges and payments: a double or wrong charge, a failed payment, invoices, promotion codes\n- product: questions about products: size, materials, stock, assembly, care and warranty, before or after buying\n- account: the customer's Larkfield account: login, personal details, deleting the account, and someone else using the account\n- needs_person: a person must read it first: a safety problem, a legal or press threat, or a complaint that asks for a manager"

def prompt(text):
    return tok.apply_chat_template([{"role": "system", "content": SYSTEM}, {"role": "user", "content": "Message:\n" + text}],
                                   tokenize=False, add_generation_prompt=True, enable_thinking=False)
print(prompt(test["text"].iloc[0])[-300:])

## 2. Score each route

For each route, we add its name (and the end of the answer) after the prompt, and add up the **log-probabilities** of its tokens: how likely the model finds each next token. A log-probability is a negative number; closer to 0 is more likely. Then we turn the seven scores into probabilities that add up to 1 (a softmax).

The model never writes anything: it cannot answer "Billing, I think", and every route gets a number.

> **Predict.** For the first test message, which route will get the highest probability? Read the message first.

In [ ]:
ROUTES = rt.ROUTES
END = tok.convert_tokens_to_ids("<|im_end|>")
LABEL_IDS = [tok(r, add_special_tokens=False).input_ids + [END] for r in ROUTES]

@torch.no_grad()
def route_scores(text):
    """The log-probability of each route name after the prompt (one pass for the prompt, one for the 7 names)."""
    ids = tok(prompt(text), return_tensors="pt").input_ids
    out = lm(input_ids=ids, use_cache=True)
    first = torch.log_softmax(out.logits[0, -1], -1)
    scores = []
    for label in LABEL_IDS:
        s = first[label[0]].item()
        if len(label) > 1:
            more = lm(input_ids=torch.tensor([label[:-1]]), past_key_values=__import__("copy").deepcopy(out.past_key_values))
            logp = torch.log_softmax(more.logits[0], -1)
            s += sum(logp[i, label[i + 1]].item() for i in range(len(label) - 1))
        scores.append(s)
    return np.array(scores)

def softmax(x, temperature=1.0):
    z = np.asarray(x) / temperature
    e = np.exp(z - z.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

first = test.iloc[0]
p = softmax(route_scores(first["text"]))
print(first["text"])
print("label:", first["route"])
print(pd.Series(p, index=ROUTES).round(3).sort_values(ascending=False).to_string())

> **Check.** You should see seven probabilities that add up to 1, with `product` first at 0.525. The label is `delivery`: the model is wrong, and sure of it.

## 3. 100 test messages

The next cell scores 100 test messages (about 1 seconds per message on a laptop CPU).

In [ ]:
sample = test.groupby("route").head(15).head(100)
t0 = time.time()
logp = np.stack([route_scores(t) for t in sample["text"]])
print(f"{(time.time() - t0) / len(sample):.2f} seconds per message")
p100 = softmax(logp)
pred = np.array(ROUTES)[p100.argmax(1)]
print("accuracy on these 100:", (pred == sample["route"].to_numpy()).mean())

> **Check.** You should see an accuracy of about 0.27 on these 100 messages. Your time per message depends on your computer.

This is the honest result of this lesson: a 0.6-billion-parameter model is **too small** to route Larkfield's messages. On all 2,000 test messages it was right on 19.4% of them with route scores (the course's runs), and 28.2% when it wrote a JSON answer. Scoring the routes gives you a probability for every route; it does not make a weak model strong. The method is what you keep: the same code works with a larger model.

## 4. Are the probabilities true?

The course team ran the same scoring on all 2,000 validation and 2,000 test messages (`data/predictions/local-llm-scores.csv`). A probability is **calibrated** when, among the messages with probability 0.8, about 80% are right. The **reliability table** groups the messages by probability; the **expected calibration error** (ECE) is the mean gap, weighted by the number of messages.

> **Predict.** Is a model that is 0.95 sure of most messages well calibrated?

In [ ]:
raw = pd.read_csv("data/predictions/local-llm-scores.csv").set_index("message_id")
meta = json.load(open("data/predictions/local-llm-scores.json"))
print(meta["model"])
t = raw.loc[test["message_id"]]
table, ece = rt.reliability(t["confidence"], t["prediction"].to_numpy() == test["route"].to_numpy())
print("ECE after temperature scaling:", ece)
table

> **Check.** You should see an ECE of 0.111, and most messages in the 0.2–0.3 bin: after scaling, the model says "I am not sure", which is true.

The file holds the probabilities **after temperature scaling**: one number (the temperature, here 5.0) divides all scores before the softmax. It was chosen on the validation messages so that the probabilities match the accuracy best. Before scaling, the ECE was 0.39. Temperature scaling does not change which route wins; it changes how sure the model says it is.

## 5. Your turn: compare with the fine-tuned encoder

> **Your turn.** Compute the ECE of the fine-tuned encoder on the test messages, from `data/predictions/encoder-finetuned.csv` (columns `prediction` and `confidence`). Put it in `ece_encoder`.

In [ ]:
enc = pd.read_csv("data/predictions/encoder-finetuned.csv").set_index("message_id").loc[test["message_id"]]
_, ece_encoder = rt.reliability(enc["confidence"], enc["prediction"].to_numpy() == test["route"].to_numpy())
print(ece_encoder)

In [ ]:
expect('ece_encoder', ece_encoder, 0.0313, tolerance=0.001)

## 6. Failure case: free text from the same model

Without the scoring, the same model writes its answer. The course team asked it, with the same prompt, to write the route for 200 test messages: 93% of the answers were exactly a route name. The others were a route with extra text, such as `- delivery` (the prompt lists the routes with dashes, and the model copied the dash). A router must check every answer, or send it to a person.

There are two fixes. **Score the options**, as in this notebook. Or **constrain the output** (constrained decoding, for example with the Outlines library): at each step, the model may only choose tokens that keep the answer on one of the route names. In the course's runs, Outlines gave a valid route for 100% of 500 test messages, and the right route for 12.8% of them. A valid answer is not a right answer. The lesson page shows both methods.

## Next

The [lesson page](https://learning.nextia-ai.com/courses/text-routing/m06/ask-for-a-decision-not-for-text/) has the diagrams and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Purpose-built decision models](https://learning.nextia-ai.com/courses/text-routing/m06/purpose-built-decision-models/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/text-routing/m06/ask-for-a-decision-not-for-text/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.